In [1]:
import numpy as np

In [7]:
# Each row represents one lap.
# Columns: tyre temperature, fuel load, track temperature

A = np.array([
    [90, 40, 30],
    [92, 38, 31],
    [88, 45, 29],
    [95, 30, 32],
    [91, 35, 30],
    [87, 48, 28],
    [94, 32, 33],
    [89, 42, 29]
], dtype=float)

b = np.array([
    91.2, 90.5, 92.1, 89.8,
    90.3, 93.0, 89.5, 91.7
], dtype=float)

# Represent b as a column vector
b = b.reshape(-1, 1)

print("Shape of A: ", A.shape)
print("Shape of b: ", b.shape)
print("First Three observations: \n", A[:3])
print("Actual lap time: \n", b[:3])

Shape of A:  (8, 3)
Shape of b:  (8, 1)
First Three observations: 
 [[90. 40. 30.]
 [92. 38. 31.]
 [88. 45. 29.]]
Actual lap time: 
 [[91.2]
 [90.5]
 [92.1]]


In [16]:
# Getting AtA and Atb

ATA = A.T @ A
print("A^T A: \n", ATA)

ATb = A.T @ b
print("A^T b: \n", ATb)

print("Shape of A^T A: ", ATA.shape)
print("Shape of A^T b: ", ATb.shape)



A^T A: 
 [[65940. 28013. 21993.]
 [28013. 12286.  9311.]
 [21993.  9311.  7340.]]
A^T b: 
 [[66052.4]
 [28265.4]
 [22011.8]]
Shape of A^T A:  (3, 3)
Shape of A^T b:  (3, 1)


In [17]:
from numpy import linalg as lng

In [19]:
weights, residuals, rank, singular_values = lng.lstsq(
    A, b, rcond = None
) 

print("Weights: \n", weights)
print("Rank of A: \n", rank)

Weights: 
 [[ 1.07393941]
 [ 0.46364041]
 [-0.80712607]]
Rank of A: 
 3


In [20]:
predicted = A @ weights
errors = b - predicted

print("Actual | Predicted | Error")

for actual, pred, err in zip(
    b.ravel(), predicted.ravel(), errors.ravel()
):
    print(f"{actual: 6.2f} | {pred: 9.2f} | {err: 7.3f}")

Actual | Predicted | Error
 91.20 |     90.99 |   0.214
 90.50 |     91.40 |  -0.900
 92.10 |     91.96 |   0.136
 89.80 |     90.11 |  -0.305
 90.30 |     89.74 |   0.558
 93.00 |     93.09 |  -0.088
 89.50 |     89.15 |   0.348
 91.70 |     91.65 |   0.053


In [23]:
squared_errors = float(np.sum(errors ** 2))

print("Sum of squared errors: ", squared_errors)
print("Mean Squared errors: ", squared_errors / len(b))

Sum of squared errors:  1.4103398444715827
Mean Squared errors:  0.17629248055894783


In [24]:
orthogonality = A.T @ errors

print("A transpose times errors : \n", orthogonality)

A transpose times errors : 
 [[4.80895324e-11]
 [2.13304929e-11]
 [1.59872116e-11]]


###Next: add an intercept to our F1 model

In [27]:
# Add a column of ones for the intercept
A_new = np.column_stack((np.ones(A.shape[0]), A))

print("New Shape: ", A_new.shape)
print(A_new)

New Shape:  (8, 4)
[[ 1. 90. 40. 30.]
 [ 1. 92. 38. 31.]
 [ 1. 88. 45. 29.]
 [ 1. 95. 30. 32.]
 [ 1. 91. 35. 30.]
 [ 1. 87. 48. 28.]
 [ 1. 94. 32. 33.]
 [ 1. 89. 42. 29.]]


In [33]:
weights_new, residual_new, rank_new, singular_values_new = (
    np.linalg.lstsq(A_new, b, rcond=None)
)

predicted_new = A_new @ weights_new
errors_new = b - predicted_new

print("New Weights [intercept, tyre, fuel, track]:")
print(weights_new)

new_squared_errors = np.sum(errors_new ** 2)
print("Sum of squared errors: \n",new_squared_errors )
print("Mean squared errors: \n", new_squared_errors / len(b))

New Weights [intercept, tyre, fuel, track]:
[[73.64174936]
 [ 0.23382947]
 [ 0.19796031]
 [-0.38083425]]
Sum of squared errors: 
 0.2343954428519
Mean squared errors: 
 0.0292994303564875


In [ ]:
new_orthogonality = A_new.T @ errors_new
print("Orthogonality remains 0: \n",new_orthogonality)

[[2.70006240e-13]
 [2.44142484e-11]
 [1.06581410e-11]
 [8.12860890e-12]]


In [35]:
# Comparing both errors..
# With and without y-intercept

print("Sum of Sq errors without intercept: ", np.sum(errors ** 2))
print("Mean of Sq errors without intercept: ", np.mean(errors ** 2))

print("Sum of Sq errors with intercept: ", np.sum(errors_new ** 2))
print("Mean of Sq errors with intercept: ", np.mean(errors_new ** 2))

Sum of Sq errors without intercept:  1.4103398444715827
Mean of Sq errors without intercept:  0.17629248055894783
Sum of Sq errors with intercept:  0.2343954428519
Mean of Sq errors with intercept:  0.0292994303564875


# F1 Lap-Time Prediction Using Least Squares

## 1. Objective

Apply linear algebra and NumPy to build a linear regression model that predicts lap times using three input features:

- Tyre temperature
- Fuel load
- Track temperature

**Dataset:** 8 synthetic observations  
**Tools:** Python, NumPy, Jupyter Notebook

> The dataset is synthetic and is used to demonstrate least-squares regression, not to model actual Formula 1 performance.

## 2. Mathematical Concepts

### Design Matrix and Target Vector

Let \(A\) represent the feature matrix and \(b\) the actual lap times.

- \(A\) initially has shape \(8\times3\).
- \(b\) has shape \(8\times1\).
- \(\hat{x}\) contains the learned model weights.

### Normal Equations

The least-squares solution satisfies:

\[
A^TA\hat{x}=A^Tb
\]

When \(A^TA\) is invertible:

\[
\hat{x}=(A^TA)^{-1}A^Tb
\]

In practice, I used `np.linalg.lstsq()` to find the solution without explicitly calculating a matrix inverse.

### Predictions and Residuals

Predictions:

\[
\hat{b}=A\hat{x}
\]

Residuals:

\[
e=b-\hat{b}
\]

A positive residual means the actual lap time was longer than predicted; a negative residual means it was shorter.

### Sum of Squared Errors

\[
SSE=\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

Mean squared error:

\[
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

Least squares minimises the SSE for the chosen linear model.

### Residual Orthogonality

A fundamental property of least-squares regression is:

\[
\boxed{A^Te\approx0}
\]

Since \(e=b-A\hat{x}\), this is equivalent to the normal equations. Geometrically, the residual is perpendicular to every column of \(A\), and \(A\hat{x}\) is the orthogonal projection of \(b\) onto the column space of \(A\).

I verified this numerically using:

```python
A.T @ errors
```

The results were approximately zero, with tiny floating-point rounding errors.

## 3. Understanding the Intercept

Without an intercept:

\[
\hat{y}=w_1T_{\text{tyre}}+
w_2F_{\text{fuel}}+
w_3T_{\text{track}}
\]

With an intercept:

\[
\hat{y}=c+w_1T_{\text{tyre}}+
w_2F_{\text{fuel}}+
w_3T_{\text{track}}
\]

I added a column of ones to the design matrix:

```python
A_new = np.column_stack((np.ones(A.shape[0]), A))
```

This changed the matrix shape from `(8, 3)` to `(8, 4)`.

The intercept allows the model to learn a constant baseline. Its value represents the prediction when all features are zero, although that may not be physically meaningful for this dataset.

## 4. Model Comparison

| Metric | Without intercept | With intercept |
|---|---:|---:|
| SSE | 1.410340 | 0.234395 |
| MSE | 0.176292 | 0.029299 |

Adding an intercept reduced training SSE by approximately **83.4%**.

This demonstrates a better fit to the synthetic training observations. It does not establish that the model will predict unseen lap times more accurately.

## 5. NumPy Skills Practised

| Operation | Purpose |
|---|---|
| `A.T` | Transpose a matrix |
| `A.T @ A` | Calculate the Gram matrix |
| `A.T @ b` | Calculate the right-hand side of the normal equations |
| `np.linalg.lstsq()` | Find least-squares weights |
| `A @ weights` | Generate predictions |
| `b - predicted` | Calculate residuals |
| `errors ** 2` | Square residuals |
| `np.sum(errors ** 2)` | Calculate SSE |
| `np.mean(errors ** 2)` | Calculate MSE |
| `.ravel()` | Flatten an array to one dimension |
| `zip()` | Iterate over corresponding values |
| `np.column_stack()` | Add a column to a matrix |

## 6. How to Improve the Model

**A. Evaluate predictions properly**

Calculate:

MAE: average absolute error, in seconds.

RMSE: square root of MSE, in seconds.

(R^2): performance relative to a mean-prediction baseline.

Plot actual versus predicted lap times and inspect residuals.

**B. Use better data**

Replace the eight synthetic observations with a larger, reliable dataset. Potential features include tyre compound, tyre age, fuel load, weather, circuit characteristics, and traffic.

**C. Test generalisation**

Split a sufficiently large dataset into training and test sets. Fit on training data and evaluate on unseen observations. For sequential lap data, consider chronological or session-based splits to reduce data leakage.

**D. Compare models**

After establishing a baseline, test feature scaling where appropriate, polynomial features, ridge regression, and other models if justified by validation results.

More complex models are not automatically better. Compare them using the same evaluation procedure.

## 7. Future Roadmap

Calculate MAE and RMSE.

Plot actual versus predicted values and residuals.

Learn feature scaling and preprocessing.

Learn training, validation, and test splits.

Repeat the experiment using real motorsport data.

Compare linear regression with regularised and nonlinear models.

Document results, limitations, and conclusions in GitHub.

## 8. Key Takeaways

Linear algebra underpins least-squares regression.

NumPy can fit a model and verify its mathematical properties.

Residual orthogonality connects projections to machine learning.

An intercept gives the model additional flexibility.

Lower training error does not guarantee better predictions on unseen data.

Data quality, useful features, and validation are essential for real-world accuracy.

**Next milestone:** Evaluate the models using MAE and RMSE, then repeat the experiment with a larger, real motorsport dataset.

**Career relevance:** This practical is an introductory step towards data-driven performance analysis in Formula 1, aerospace, and autonomous systems.